## Phase 3 — Lucas-Kanade Optical Flow Tracking
## Tracking Harris corners between consecutive frames + point filtering
**Ali | 23L-2619 | BS Data Science 6th Semester | Fundamentals of Computer Vision | Final Project**

In [1]:
import sys, os, cv2
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = r"C:\Users\pakistan\OneDrive\Desktop\cv_project"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["core","utils","metrics","visualization"]):
        del sys.modules[mod]

from utils.config_loader import ConfigLoader
from core.harris_detector import HarrisDetector
from core.lucas_kanade import LKTracker

cfg    = ConfigLoader("config.yaml").config
harris = HarrisDetector(cfg)
lk     = LKTracker(cfg)

print("✅ Setup complete")

✅ Setup complete


In [2]:
import glob
import os
import sys

from metrics.logger import AppLogger
AppLogger.reset()

pyc_files = glob.glob("../metrics/__pycache__/*.pyc")
for pyc in pyc_files:
    if os.path.exists(pyc):
        os.remove(pyc)

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["core", "utils", "metrics", "visualization"]):
        del sys.modules[mod]

In [3]:
cap = cv2.VideoCapture(cfg.input.source)

# Skip to a frame with clear motion
cap.set(cv2.CAP_PROP_POS_FRAMES, 30)
ret1, frame1 = cap.read()
ret2, frame2 = cap.read()
cap.release()

gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(frame1, cv2.COLOR_BGR2RGB))
axes[0].set_title("Frame N (prev)", fontsize=12)
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(frame2, cv2.COLOR_BGR2RGB))
axes[1].set_title("Frame N+1 (curr)", fontsize=12)
axes[1].axis("off")
plt.suptitle("Two Consecutive Frames for LK Tracking", fontsize=13, fontweight="bold")
plt.tight_layout()

os.makedirs(r"outputs\phase_3", exist_ok=True)
save_path = r"outputs\phase_3\lk_tracking.png"
fig.savefig(save_path, bbox_inches="tight", dpi=100)
plt.close(fig)
print(f"Plot saved to: {save_path}")

# Restore if kernel was reset
quality_levels = [0.001, 0.005, 0.01, 0.05, 0.1]
counts = counts if 'counts' in dir() else []

for ql, c in zip(quality_levels, counts):
    print(f"quality_level={ql:.3f}  →  {c} corners")

Plot saved to: outputs\phase_3\lk_tracking.png


In [4]:
# ── Load frame index and bbox from config (defined once in Phase 2) ──
frame_idx = cfg.input.start_frame
bbox      = tuple(cfg.input.bbox)
x, y, w, h = bbox

# Load chosen frame and the next frame from video
cap_load = cv2.VideoCapture(cfg.input.source)
cap_load.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
_, frame1 = cap_load.read()
_, frame2 = cap_load.read()   # consecutive next frame
cap_load.release()

chosen_frame     = frame1
chosen_frame_idx = frame_idx

gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

# Harris detection on chosen frame
mask   = harris.create_bbox_mask(gray1.shape, bbox)
points = harris.detect(gray1, mask=mask)

print(f"Frame pair         : {frame_idx} → {frame_idx + 1}")
print(f"Bbox (full res)    : x={x}, y={y}, w={w}, h={h}")
print(f"Harris corners     : {len(points)}")

Frame pair         : 0 → 1
Bbox (full res)    : x=2112, y=1728, w=216, h=306
Harris corners     : 201


In [5]:
good_new, good_prev, final_mask = lk.track(gray1, gray2, points)

print(f"Points before LK  : {len(points)}")
print(f"Points after LK   : {len(good_new)}")
print(f"Points lost       : {len(points) - len(good_new)}")
print(f"Survival rate     : {len(good_new)/max(len(points),1)*100:.1f}%")

Points before LK  : 201
Points after LK   : 201
Points lost       : 0
Survival rate     : 100.0%


In [6]:
vis = cv2.cvtColor(frame2.copy(), cv2.COLOR_BGR2RGB)

# Draw bounding box
cv2.rectangle(vis, (x,y), (x+w, y+h), (0,220,0), 2)

# Draw flow vectors (prev → new)
for p0, p1 in zip(good_prev, good_new):
    x0, y0 = int(p0[0,0]), int(p0[0,1])
    x1, y1 = int(p1[0,0]), int(p1[0,1])
    cv2.line(vis, (x0,y0), (x1,y1), (255,100,0), 1, cv2.LINE_AA)
    cv2.circle(vis, (x1,y1), 4, (255,255,0), -1, cv2.LINE_AA)

plt.figure(figsize=(12,6))
plt.imshow(vis)
plt.title(f"LK Optical Flow — {len(good_new)} tracked points (arrows show motion)", fontsize=13)
plt.axis("off")
plt.tight_layout()
os.makedirs(r"outputs\phase_3", exist_ok=True)
save_path = r"outputs\phase_3\lk_tracking.png"
plt.savefig(save_path, bbox_inches="tight", dpi=100)
plt.close()
print(f"Plot saved to: {save_path}")

Plot saved to: outputs\phase_3\lk_tracking.png


In [7]:
# Manually compute FB errors for all original points to show distribution
lk_kw = dict(
    winSize=tuple(cfg.lucas_kanade.win_size),
    maxLevel=cfg.lucas_kanade.max_level,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
              cfg.lucas_kanade.max_iter, cfg.lucas_kanade.epsilon)
)
next_pts, st_fwd, _ = cv2.calcOpticalFlowPyrLK(gray1, gray2, points, None, **lk_kw)
back_pts, st_bwd, _ = cv2.calcOpticalFlowPyrLK(gray2, gray1, next_pts, None, **lk_kw)
fb_errors = np.linalg.norm(points.reshape(-1,2) - back_pts.reshape(-1,2), axis=1)

thresh = cfg.lucas_kanade.fb_error_threshold
kept   = (fb_errors < thresh).sum()
lost   = (fb_errors >= thresh).sum()

plt.figure(figsize=(10,4))
plt.hist(fb_errors, bins=30, color="#00cc66", edgecolor="white", alpha=0.85)
plt.axvline(thresh, color="red", linestyle="--", linewidth=2,
            label=f"FB threshold = {thresh}px")
plt.xlabel("Forward-Backward Error (pixels)", fontsize=11)
plt.ylabel("Point count", fontsize=11)
plt.title("Forward-Backward Error Distribution", fontsize=13, fontweight="bold")
plt.legend(fontsize=11)
plt.tight_layout()
os.makedirs(r"outputs\phase_3", exist_ok=True)
save_path = r"outputs\phase_3\fb_error_distribution.png"
plt.savefig(save_path, bbox_inches="tight", dpi=100)
plt.close()
print(f"Plot saved to: {save_path}")

print(f"FB threshold : {thresh} px")
print(f"Points kept  : {kept}  (error < threshold)")
print(f"Points removed: {lost} (error >= threshold)")

Plot saved to: outputs\phase_3\fb_error_distribution.png
FB threshold : 3.5 px
Points kept  : 201  (error < threshold)
Points removed: 0 (error >= threshold)


In [8]:
# Show how median displacement shifts the bbox
displacements = good_new[:,0,:] - good_prev[:,0,:]
dx, dy = np.median(displacements, axis=0)

new_bbox = (int(round(x+dx)), int(round(y+dy)), w, h)
nx, ny   = new_bbox[0], new_bbox[1]

vis2 = cv2.cvtColor(frame2.copy(), cv2.COLOR_BGR2RGB)
cv2.rectangle(vis2, (x,y),   (x+w,   y+h),   (100,100,255), 2)  # old bbox (blue)
cv2.rectangle(vis2, (nx,ny), (nx+w, ny+h),   (0,220,0),     2)  # new bbox (green)

plt.figure(figsize=(12,6))
plt.imshow(vis2)
plt.title(f"Bbox Update — blue=previous, green=updated  |  shift: dx={dx:.1f}px, dy={dy:.1f}px", fontsize=12)
plt.axis("off")
plt.tight_layout()
os.makedirs(r"outputs\phase_3", exist_ok=True)
save_path = r"outputs\phase_3\bbox_update.png"
plt.savefig(save_path, bbox_inches="tight", dpi=100)
plt.close()
print(f"Plot saved to: {save_path}")

print(f"Median displacement : dx={dx:.2f}px, dy={dy:.2f}px")
print(f"Old bbox : {bbox}")
print(f"New bbox : {new_bbox}")

Plot saved to: outputs\phase_3\bbox_update.png
Median displacement : dx=-0.72px, dy=-1.97px
Old bbox : (2112, 1728, 216, 306)
New bbox : (2111, 1726, 216, 306)


In [ ]:
## Summary
#- LK optical flow tracks each Harris corner from frame N to frame N+1
#- Forward-backward error check filters unreliable points (round-trip error > threshold)
#- Median displacement of all surviving points shifts the bounding box robustly
#- Median is used instead of mean to suppress the effect of outlier point movements